# Sample Notebook: OLMoEarth Pixel-Level Embeddings

This notebook shows how to produce **spatial (pixel-level) embeddings** from OLMoEarth using the lower-level encoder output.

Current Pixelverse wrapper `forward(...)` returns patch-level embeddings `(B, H, W, D)` by default (`patch_size=1`). This notebook still shows the lower-level encoder path for explicit control over spatial pooling and token outputs.


In [ ]:
import numpy as np
import torch
import xarray as xr
from einops import rearrange
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.nn.flexi_vit import PoolingType
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample

import pixelverse as pv

In [ ]:
MODEL_NAME = "olmoearth_nano"
model, transforms = pv.create_model(MODEL_NAME)
weights = pv.get_weights(MODEL_NAME)
model.eval()
weights.meta

## Output shape

With `patch_size=8`, pixel-level embeddings are produced on a coarser grid:
- input: `(H, W)`
- output: `(H / 8, W / 8)` (assuming divisible by patch size)

For `olmoearth_nano`, the embedding dimension is `128`.


In [ ]:
OLMOEARTH_BANDS = [
    "B02",
    "B03",
    "B04",
    "B08",
    "B05",
    "B06",
    "B07",
    "B8A",
    "B11",
    "B12",
    "B01",
    "B09",
]

S2_COMMON_NAME_TO_OLMOEARTH = {
    "coastal": "B01",
    "blue": "B02",
    "green": "B03",
    "red": "B04",
    "nir": "B08",
    "rededge1": "B05",
    "rededge2": "B06",
    "rededge3": "B07",
    "nir08": "B8A",
    "swir16": "B11",
    "swir22": "B12",
    "nir09": "B09",
}

In [ ]:
def build_olmoearth_inputs_from_xarray(ds: xr.Dataset) -> tuple[torch.Tensor, torch.Tensor]:
    # Accept datasets from pixelverse.download.sentinel2.get_s2_time_series (common-name vars)
    # or datasets already renamed to OLMoEarth band ids.
    if set(S2_COMMON_NAME_TO_OLMOEARTH).issubset(ds.data_vars):
        ds = ds.rename(S2_COMMON_NAME_TO_OLMOEARTH)

    missing = [band for band in OLMOEARTH_BANDS if band not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required bands: {missing}")

    arr = ds[OLMOEARTH_BANDS].to_array(dim="band").transpose("time", "band", "y", "x").values
    x = torch.from_numpy(arr).unsqueeze(0).float()  # (B, T, C, H, W)

    t = ds.time.dt
    timestamps_np = np.stack(
        [
            t.day.values.astype(np.int64),
            (t.month.values - 1).astype(np.int64),  # zero-indexed months for OLMoEarth
            t.year.values.astype(np.int64),
        ],
        axis=-1,
    )
    timestamps = torch.from_numpy(timestamps_np).unsqueeze(0)  # (B, T, 3)
    return x, timestamps

## Synthetic runnable example

This runs without STAC download and demonstrates pixel-level output shape.


In [ ]:
B, T, C, H, W = 1, 4, 12, 16, 16
x = torch.randint(0, 10000, (B, T, C, H, W), dtype=torch.int32).float()
timestamps = torch.tensor(
    [[[15, 0, 2024], [15, 1, 2024], [15, 2, 2024], [15, 3, 2024]]],
    dtype=torch.long,
)

with torch.no_grad():
    x_norm = transforms(x)
    x_olmo = rearrange(x_norm, "b t c h w -> b h w t c").contiguous()
    sample = MaskedOlmoEarthSample(
        timestamps=timestamps,
        sentinel2_l2a=x_olmo,
        sentinel2_l2a_mask=torch.zeros(
            (B, H, W, T, model.num_bandsets),
            dtype=torch.long,
            device=x.device,
        ),
    )
    out = model.encoder(
        sample,
        patch_size=model.patch_size,
        input_res=model.input_res,
        fast_pass=model.fast_pass,
    )
    tokens_and_masks = out["tokens_and_masks"]
    pixel_embeddings = tokens_and_masks.pool_spatially(PoolingType.MEAN)

pixel_embeddings.shape

`pixel_embeddings` shape is `(B, H_patch, W_patch, D)`. Convert to `xarray` if you want feature-first ordering for storage or plotting.


In [ ]:
pixel_embeddings_da = xr.DataArray(
    pixel_embeddings[0].cpu().numpy(),
    dims=["y_patch", "x_patch", "feature"],
).transpose("feature", "y_patch", "x_patch")
pixel_embeddings_da

## Example with `get_s2_time_series`

`get_s2_time_series(...)` accepts a `bands=` argument. For OLMoEarth, read the required STAC band names from the model weights metadata.

```python
from pixelverse.download.sentinel2 import get_s2_time_series

weights = pv.get_weights("olmoearth_nano")
ds = get_s2_time_series(
    bbox=...,
    year=2024,
    bands=weights.meta["s2_stac_bands"],
)
x, timestamps = build_olmoearth_inputs_from_xarray(ds)
```
